In [1]:
# word2vec

In [2]:
import numpy as np
import pandas as pd

In [3]:
imdb = pd.read_csv(r"C:\Users\User\Downloads\IMDB-Dataset (1).csv")
imdb.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
y = imdb['sentiment'].map({'positive' : 1, 'negative' :0}).values

In [5]:
y

array([1, 1, 1, ..., 0, 0, 0], shape=(50000,))

In [6]:
print('number of rows', len(imdb))
print(len(imdb.columns))
print(imdb.columns)

number of rows 50000
2
Index(['review', 'sentiment'], dtype='object')


In [7]:
print(imdb.isnull().sum())

review       0
sentiment    0
dtype: int64


In [8]:
print(imdb['sentiment'].value_counts())

sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [9]:
# we need to clean some html tags


In [10]:
import re

def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    
    # Remove HTML tags such as <br />
    text = re.sub(r'<.*?>', ' ', text)
    
    # Keep only letters and spaces
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [12]:
imdb['clean_review'] = imdb['review'].apply(clean_text)

In [13]:
print("Original:")
print(imdb['review'].iloc[1])

print("\nCleaned:")
print(imdb['clean_review'].iloc[1])

Original:
A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master's of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional 'dream' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell's murals decorating every surface) are terribly we

In [14]:
from gensim.utils import simple_preprocess
words = []
for review in imdb['clean_review']:
    tokens = simple_preprocess(review)
    words.append(tokens)

In [15]:
print("Number of reviews:", len(words))

print("\nFirst review tokens:")
print(words[0])

print("\nSecond review tokens:")
print(words[1])

Number of reviews: 50000

First review tokens:
['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'you', 'll', 'be', 'hooked', 'they', 'are', 'right', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with', 'me', 'the', 'first', 'thing', 'that', 'struck', 'me', 'about', 'oz', 'was', 'its', 'brutality', 'and', 'unflinching', 'scenes', 'of', 'violence', 'which', 'set', 'in', 'right', 'from', 'the', 'word', 'go', 'trust', 'me', 'this', 'is', 'not', 'show', 'for', 'the', 'faint', 'hearted', 'or', 'timid', 'this', 'show', 'pulls', 'no', 'punches', 'with', 'regards', 'to', 'drugs', 'sex', 'or', 'violence', 'its', 'is', 'hardcore', 'in', 'the', 'classic', 'use', 'of', 'the', 'word', 'it', 'is', 'called', 'oz', 'as', 'that', 'is', 'the', 'nickname', 'given', 'to', 'the', 'oswald', 'maximum', 'security', 'state', 'penitentary', 'it', 'focuses', 'mainly', 'on', 'emerald', 'city', 'an', 'experimental', 'section', 'of', 'the', 'p

In [16]:
print("Original reviews:", len(imdb))
print("Tokenized reviews:", len(words))

Original reviews: 50000
Tokenized reviews: 50000


In [17]:
from gensim.models import Word2Vec
word2vec_model = Word2Vec(
    sentences = words,
    vector_size = 100,
    window = 5,
    min_count = 2,
    workers = 4,
    epochs = 10
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

In [18]:
print(len(word2vec_model.wv.index_to_key))
print(word2vec_model.vector_size)

61963
100


In [19]:
print(word2vec_model is not None)

True


In [22]:
import numpy as np
def avg_word2vec(doc):
    vectors = [
        word2vec_model.wv[word]
        for word in doc
        if word in word2vec_model.wv.key_to_index]
    if len(vectors)==0:
        return np.zeros(word2vec_model.vector_size)
    return np.mean(vectors ,  axis = 0)

In [23]:
x = []
for doc in words:
    x.append(avg_word2vec(doc))
x = np.array(x)

In [25]:
print("X shape:", x.shape)
print("Number of reviews:", len(words))

X shape: (50000, 100)
Number of reviews: 50000


In [26]:
y = imdb['sentiment'].map({
    'positive': 1,
    'negative': 0
}).values

In [27]:
print("Y shape:", y.shape)
print("Positive:", np.sum(y == 1))
print("Negative:", np.sum(y == 0))

Y shape: (50000,)
Positive: 25000
Negative: 25000


In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (40000, 100)
X_test : (10000, 100)
y_train: (40000,)
y_test : (10000,)


In [30]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [31]:
y_pred = lr_model.predict(X_test)

print("Actual labels:   ", y_test[:20])
print("Predicted labels:", y_pred[:20])

Actual labels:    [0 0 1 0 0 0 0 0 0 0 0 1 1 0 0 1 1 1 0 1]
Predicted labels: [0 1 1 0 0 0 0 0 0 0 0 1 1 0 1 1 1 1 1 1]


In [32]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test , y_pred)
print("Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)


Accuracy: 0.86
Accuracy (%): 86.0


In [33]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=['negative', 'positive']
))

              precision    recall  f1-score   support

    negative       0.86      0.86      0.86      5000
    positive       0.86      0.86      0.86      5000

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000

